In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

## Hypothesis

Our general hypothesis is that the location of plants are disproportionately placed in marginalized communities. The characteristic of a marginalized community is not singular, which illicts the need to conduct multiple hypothesis testing to observe potential inequalities across multiple socioeconomic demographic metrics.

Our alternative hypotheses are as follows:
1) Plants are disproportionately located in areas with higher % People of Color
2) Plants are disproportionately located in areas with higher % Low Income
3) Plants are disproportionately located in areas with higher % Less Than High School Education
4) Plants are disproportionately located in areas with higher % Unemployment Rate
5) There is a statistically significant difference in the % of Age under 5 for counties with plants vs without
6) There is a statistically significant difference in the % of Age over 64 for counties with plants vs without

We will be conducting A/B testing against each hypothesis. This is because we can treat each plant in the eGRID data as the 'treatment' of having a plant. The other counties across the US will be the control groups, for not having a plant. 

To correct for the multiple hypothesis tests, we will use two different methods:
- To control the FDR at 0.05, we will use the Benjamini–Yekutieli procedure, which controls the false discovery rate under arbitrary dependence assumptions. This is needed because the demographic metrics are not independent (source).
- To control for the FWER at 0.05, we will use the Bonferroni correction.


In [3]:
counties = pd.read_csv('data/mh_analysis_ready.csv', index_col=0)
counties

,County FIPS,Plant state abbreviation,Plant county name,EPA Region,has_plant,Total Population,People of Color (%),Low Income (%),Less Than High School Education (%),Limited English Speaking (%),Unemployment Rate (%),Over Age 64 (%),Under Age 5 (%),Limited Life Expectancy (%),Plant primary fuel category_x,Plant annual net generation (MWh)
2,1001,AL,Autauga County,4,0,58239.0,26.902934,30.770029,10.415510,0.146413,2.824625,15.135905,5.697213,NaN,NaN,NaN
3,1003,AL,Baldwin County,4,0,227131.0,17.427388,25.847738,8.985844,0.837252,3.685828,20.607051,5.298704,NaN,NaN,NaN
4,1005,AL,Barbour County,4,0,25259.0,55.390158,50.314607,24.328980,1.287412,8.624186,19.007087,5.225860,NaN,NaN,NaN
5,1007,AL,Bibb County,4,0,22412.0,25.950384,40.404762,19.461917,0.324721,9.706819,16.036052,5.336427,NaN,NaN,NaN
6,1009,AL,Blount County,4,0,58884.0,14.076829,33.465357,16.351923,1.582160,6.023723,17.974322,5.887847,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8510,56009,WY,Converse,8,1,289.0,11.000000,24.000000,6.000000,0.000000,0.000000,16.000000,5.000000,18.0,WIND,"254,753"
8512,56005,WY,Campbell,8,1,1861.0,16.000000,29.000000,8.000000,0.000000,2.000000,19.000000,3.000000,19.0,COAL,"643,236"
8513,56005,WY,Campbell,8,1,1638.0,15.000000,29.000000,7.000000,0.000000,2.000000,18.000000,3.000000,19.0,COAL,"663,372"
8514,56005,WY,Campbell,8,1,1649.0,15.000000,29.000000,7.000000,0.000000,2.000000,18.000000,3.000000,19.0,COAL,"794,708"


In [4]:
# 1 row per county
counties2 =  pd.read_csv('data/mh_analysis_2.csv', index_col=0)
counties2

,County FIPS,Plant state abbreviation,Plant county name,EPA Region,Total Population,People of Color (%),Low Income (%),Less Than High School Education (%),Limited English Speaking (%),Unemployment Rate (%),Over Age 64 (%),Under Age 5 (%),has_plant
0,1001,AL,Autauga County,4.0,58239.0,26.902934,30.770029,10.415510,0.146413,2.824625,15.135905,5.697213,0.0
1,1003,AL,Baldwin County,4.0,227131.0,17.427388,25.847738,8.985844,0.837252,3.685828,20.607051,5.298704,0.0
2,1005,AL,Barbour County,4.0,25259.0,55.390158,50.314607,24.328980,1.287412,8.624186,19.007087,5.225860,0.0
3,1007,AL,Bibb County,4.0,22412.0,25.950384,40.404762,19.461917,0.324721,9.706819,16.036052,5.336427,0.0
4,1009,AL,Blount County,4.0,58884.0,14.076829,33.465357,16.351923,1.582160,6.023723,17.974322,5.887847,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9440,56037,WY,Sweetwater County,8.0,42459.0,21.705645,22.398131,7.294363,1.976946,6.713718,12.694599,6.257802,1.0
9442,56039,WY,Teton County,8.0,23319.0,20.073760,21.523236,3.969647,4.889309,2.402089,15.476650,4.579956,1.0
9443,56041,WY,Uinta County,8.0,20514.0,12.971629,25.687078,6.430892,1.641694,3.467517,14.731403,6.887979,1.0
9445,56043,WY,Washakie County,8.0,7768.0,18.357364,25.391808,5.900793,0.296736,2.519960,22.270855,5.136457,0.0


In [18]:
numerical_col = 'People of Color (%)' #, 'Low Income (%)', 'Less Than High School Education (%)', 'Unemployment Rate (%)' , 'Over Age 64 (%)', 'Under Age 5 (%)']
binary_col = 'has_plant'

shuffled_table = counties2.copy()
    
shuffled_labels = counties2.sample(replace=False, frac = 1, random_state = 1)[binary_col]
shuf = shuffled_labels.to_numpy()
shuffled_table['Shuffled Label'] = shuf
selected_shuf = shuffled_table.loc[:, (numerical_col, 'Shuffled Label')]

series = selected_shuf.groupby('Shuffled Label').mean().loc[:, numerical_col]
dif = series.iloc[1] - series.iloc[0]

selected = counties2.loc[:, (numerical_col, binary_col)]
series_obs = selected.groupby(binary_col).mean().loc[:, numerical_col] #iloc[1] is has_plant == 1, iloc[0] is has_plant == 0
observed_difference = series_obs.iloc[1] - series_obs.iloc[0] # one sided alternative hypothesis
observed_difference

7.159251514138102

In [36]:
shuffled_table


,County FIPS,Plant state abbreviation,Plant county name,EPA Region,Total Population,People of Color (%),Low Income (%),Less Than High School Education (%),Limited English Speaking (%),Unemployment Rate (%),Over Age 64 (%),Under Age 5 (%),has_plant,Shuffled Label
0,1001,AL,Autauga County,4.0,58239.0,26.902934,30.770029,10.415510,0.146413,2.824625,15.135905,5.697213,0.0,0.0
1,1003,AL,Baldwin County,4.0,227131.0,17.427388,25.847738,8.985844,0.837252,3.685828,20.607051,5.298704,0.0,0.0
2,1005,AL,Barbour County,4.0,25259.0,55.390158,50.314607,24.328980,1.287412,8.624186,19.007087,5.225860,0.0,0.0
3,1007,AL,Bibb County,4.0,22412.0,25.950384,40.404762,19.461917,0.324721,9.706819,16.036052,5.336427,0.0,0.0
4,1009,AL,Blount County,4.0,58884.0,14.076829,33.465357,16.351923,1.582160,6.023723,17.974322,5.887847,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9440,56037,WY,Sweetwater County,8.0,42459.0,21.705645,22.398131,7.294363,1.976946,6.713718,12.694599,6.257802,1.0,0.0
9442,56039,WY,Teton County,8.0,23319.0,20.073760,21.523236,3.969647,4.889309,2.402089,15.476650,4.579956,1.0,0.0
9443,56041,WY,Uinta County,8.0,20514.0,12.971629,25.687078,6.430892,1.641694,3.467517,14.731403,6.887979,1.0,0.0
9445,56043,WY,Washakie County,8.0,7768.0,18.357364,25.391808,5.900793,0.296736,2.519960,22.270855,5.136457,0.0,1.0


In [8]:
def difference_of_means(table, group_label, numerical_col):
    
    series = table.groupby('Shuffled Label').mean().loc[:, numerical_col]
   
    return series.iloc[1] - series.iloc[0]

In [9]:
def one_simulated_difference_of_means(numerical_col, binary_col, i):

    shuffled_table = counties2.copy()
    
    shuffled_labels = counties2.sample(replace=False, frac = 1, random_state = i)[binary_col]
    shuf = shuffled_labels.to_numpy()
    shuffled_table['Shuffled Label'] = shuf
    selected_shuf = shuffled_table.loc[:, (numerical_col, 'Shuffled Label')]
    
    return difference_of_means(selected_shuf, 'Shuffled Label', numerical_col)   

In [22]:
def avg_difference_in_means(numerical_col, binary_col= 'has_plant'):
    """
   The function computes the p-value for a test of the following hypothesis test:
        H0 : There is no difference in the average value of numerical_col between the two
            groups specified in binary_col.
        H1 : The average value of numerical_col is different for the two groups specified
            in binary_col
    inputs
        numerical_col: a numerical column name
        binary_col: a binary column name
    """
    selected = counties2.loc[:, (numerical_col, binary_col)]
    series_obs = selected.groupby(binary_col).mean().loc[:, numerical_col]
    observed_difference = series_obs.iloc[1] - series_obs.iloc[0]

    differences = []

    repetitions = 2500
    for i in np.arange(repetitions):
        new_difference = one_simulated_difference_of_means(numerical_col, binary_col, i)
        differences = np.append(differences, new_difference)                               

    empirical_p = np.count_nonzero(differences >= observed_difference) / repetitions #how many samples have as extreme of a difference?

    #print(f' observed dif: {observed_difference}, empirical p: {empirical_p}')
    return empirical_p
    

In [ ]:
numerical_cols = ['People of Color (%)', 'Low Income (%)', 'Less Than High School Education (%)', 'Unemployment Rate (%)' , 'Over Age 64 (%)', 'Under Age 5 (%)']
binary_cols = ['has_plant']
pvals = {}
for i in numerical_cols:
    pvals[f'{i} and {j}'] = avg_difference_in_means(i)

pvals

In [20]:
fwer = 0.05
num_tests = len(numerical_cols) * len(binary_cols)
fwer_threshold = fwer / num_tests
print(f'FWER threshold: {fwer_threshold}')
reject_null = [x for x in list(pvals.keys()) if pvals[x] <= fwer_threshold]
#list(pvals.values())
reject_null

FWER threshold: 0.008333333333333333


['People of Color (%) and has_plant',
 'Less Than High School Education (%) and has_plant',
 'Unemployment Rate (%) and has_plant']

In [21]:


p_sorted = sorted(list(pvals.values()))

m = len(p_sorted)  
k = np.arange(1, m+1)  # index of each test in sorted order
alpha = 0.05
c_m = np.sum([1/i for i in range(1, m)])
compare = (alpha * k) /(m* c_m)
below_than = p_sorted <= compare
cols = {'k' : k, 'p-vals' : p_sorted, 'compare': compare, 'below than' : below_than}

ps = pd.DataFrame(cols)
ps


,k,p-vals,compare,below than
0,1,0.0000,0.003650,True
1,2,0.0000,0.007299,True
2,3,0.0024,0.010949,True
3,4,1.0000,0.014599,False
4,5,1.0000,0.018248,False
5,6,1.0000,0.021898,False


### Results
- Summarize and interpret the results from the hypothesis tests themselves.
- 